# NB_Raw_Metadata_Ingestion
Ingests files from LH_Claims/Files/raw into LH_Claims tables based on ReferenceData config.

In [ ]:
from pyspark.sql.functions import current_timestamp

# 1. Read metadata from ReferenceData lakehouse
# (Assumes NB is attached to LH_ReferenceData for metadata access)
metadata_df = spark.sql("SELECT * FROM config_ingestion_metadata WHERE active_flag = true")
sources = metadata_df.collect()

for source in sources:
    source_object = source['source_object']
    domain = source['domain']
    target_lh = source['target_lakehouse']
    target_table = source['target_table']
    
    print(f"Processing: {source_object} for {domain} (Target LH: {target_lh})")
    
    try:
        # 2. Read from raw folder in the domain lakehouse
        # Note: In Fabric, we can access other LHs via the workspace path
        raw_path = f"Files/raw/{source_object}"
        
        df = spark.read.option("header", "true").csv(raw_path)
        
        # 3. Add audit info
        df_bronze = df.withColumn("ingestion_timestamp", current_timestamp())
        
        # 4. Save to Bronze Table
        # Using default lakehouse write or explicit name if cross-lakehouse is needed
        # For this lab, we'll write to the default attached lakehouse tables.
        df_bronze.write.format("delta").mode("overwrite").saveAsTable(target_table)
        print(f"Success: {target_table} updated in domain lakehouse.")
        
    except Exception as e:
        print(f"Failed to process {source_object}: {str(e)}")